# AquaTrack: product-demand model training
This notebook is for the Sprint 2 experiment. It compares Random Forest and Gradient Boosting for 7-day and 30-day case demand across the four bottle products. A previous-period persistence estimate is an experiment-only benchmark; it is not a method shown in AquaTrack.

The data are entirely synthetic. April-September 2026 is excluded from this experiment. Earlier application runs may already have evaluated some of those dates, so do not claim that this period is an independent untouched test. Model scores here do not prove performance at a real company. Run cells from top to bottom in Google Colab using a CPU runtime.

In [ ]:
%pip -q install 'numpy>=2,<3' 'pandas>=2,<3' 'scikit-learn>=1.6,<2' 'matplotlib>=3.8,<4' 'joblib>=1.4,<2'
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, platform, shutil
from datetime import datetime, timezone
from pathlib import Path
import joblib, matplotlib, matplotlib.pyplot as plt, numpy as np, pandas as pd, sklearn
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Paste the Git commit that contains this notebook when prompted.
GIT_COMMIT = input('Git commit hash for this notebook: ').strip()
assert len(GIT_COMMIT) >= 7, 'Enter a real Git commit hash.'
DATA_DIR = Path('/content/drive/MyDrive/AquaTrack/data')
DATA_FILE = DATA_DIR / 'daily_product_demand.csv'
METADATA_FILE = DATA_DIR / 'dataset_metadata.json'
assert DATA_FILE.is_file() and METADATA_FILE.is_file(), 'Copy both dataset files to MyDrive/AquaTrack/data first.'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = Path('/content/drive/MyDrive/AquaTrack/experiments') / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / 'models').mkdir()
(RUN_DIR / 'plots').mkdir()
print('Evidence folder:', RUN_DIR)


In [ ]:
# Check the dataset before training. Do not silently replace missing dates.
metadata = json.loads(METADATA_FILE.read_text(encoding='utf-8'))
data = pd.read_csv(DATA_FILE, parse_dates=['date'])
required = {'date', 'sku', 'cases_ordered'}
assert required.issubset(data.columns), f'Missing columns: {required - set(data.columns)}'
assert not data[list(required)].isna().any().any(), 'Missing dates, products, or targets.'
assert not data.duplicated(['date', 'sku']).any(), 'Duplicate date/product rows.'
assert pd.api.types.is_numeric_dtype(data['cases_ordered']), 'Cases must be numeric.'
assert (data['cases_ordered'] >= 0).all(), 'Cases cannot be negative.'
assert (data['cases_ordered'] % 1 == 0).all(), 'Cases must be whole numbers.'
dates = pd.date_range('2024-01-01', '2026-09-30', freq='D')
products = ['WATER-500ML', 'WATER-1L', 'WATER-5L', 'WATER-20L']
assert set(data['sku']) == set(products), 'Expected exactly the four bottled-water products.'
assert len(data) == len(dates) * len(products), 'Some product/date rows are missing.'
for sku in products:
    observed = pd.DatetimeIndex(data.loc[data.sku == sku, 'date'].sort_values())
    assert observed.equals(dates), f'Incomplete dates for {sku}'
assert metadata['dataset_type'] == 'Entirely synthetic'
assert metadata['period_start'] == '2024-01-01' and metadata['period_end'] == '2026-09-30'
TRAIN_END = pd.Timestamp('2025-09-30')
VALIDATION_END = pd.Timestamp('2026-03-31')
TEST_START = pd.Timestamp('2026-04-01')
assert TRAIN_END < VALIDATION_END < TEST_START
train_last = dates.get_loc(TRAIN_END)
validation_last = dates.get_loc(VALIDATION_END)
data_hash = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
print('Rows:', len(data), '| Products:', len(products), '| Days:', len(dates))
print('Training ends:', TRAIN_END.date(), '| Validation ends:', VALIDATION_END.date())
print('April-September 2026 is excluded from this experiment.')
print('Dataset SHA-256:', data_hash)


In [ ]:
# Every feature is calculated from demand known at the forecast origin.
HORIZONS = (7, 30)
FIRST_ORIGIN = 89
SEED = 2026
ML_METHODS = ('random_forest', 'gradient_boosting')
BENCHMARK = 'previous_period'
def features(values, origin):
    history = values[:origin + 1]
    return [history[-1], history[-7], history[-14], history[-28],
            float(np.mean(history[-7:])), float(np.mean(history[-28:])),
            float(np.std(history[-28:])), origin % 7]

def make_model(method):
    if method == 'random_forest':
        return RandomForestRegressor(n_estimators=120, max_depth=5,
                                     min_samples_leaf=4, random_state=SEED)
    if method == 'gradient_boosting':
        return GradientBoostingRegressor(n_estimators=120, max_depth=2,
                                         min_samples_leaf=5, learning_rate=0.04,
                                         random_state=SEED)
    raise ValueError('Unknown model')

def examples(values, origins, horizon):
    x = np.asarray([features(values, origin) for origin in origins], dtype=float)
    y = np.asarray([sum(values[origin + 1:origin + horizon + 1])
                    for origin in origins], dtype=float)
    return x, y

prediction_rows = []
for sku in products:
    # The CSV has one row per day, including days with zero orders.
    values = data.loc[data.sku == sku].sort_values('date')['cases_ordered'].to_numpy(dtype=float)
    for horizon in HORIZONS:
        # A training target must finish by 30 September 2025.
        train_origins = list(range(FIRST_ORIGIN, train_last - horizon + 1, 7))
        # A validation target must finish by 31 March 2026.
        validation_origins = list(range(train_last, validation_last - horizon + 1, 7))
        x_train, y_train = examples(values, train_origins, horizon)
        x_valid, y_valid = examples(values, validation_origins, horizon)
        assert len(train_origins) >= 30 and len(validation_origins) >= 15
        results = {BENCHMARK: [sum(values[origin - horizon + 1:origin + 1])
                               for origin in validation_origins]}
        for method in ML_METHODS:
            fitted = make_model(method).fit(x_train, y_train)
            results[method] = fitted.predict(x_valid)
        for method, estimates in results.items():
            for origin, actual, estimate in zip(validation_origins, y_valid, estimates):
                prediction_rows.append({'sku': sku, 'horizon_days': horizon,
                    'method': method, 'origin_date': dates[origin].date().isoformat(),
                    'actual_cases': float(actual),
                    'predicted_cases': max(0.0, float(estimate))})
predictions = pd.DataFrame(prediction_rows)
predictions['error_cases'] = predictions.predicted_cases - predictions.actual_cases
predictions['absolute_error_cases'] = predictions.error_cases.abs()
print('Validation predictions:', len(predictions))
display(predictions.head())


In [ ]:
# MAE is the average miss in cases; bias shows over- or under-prediction.
metrics = predictions.groupby(['sku', 'horizon_days', 'method'], as_index=False).agg(
    validation_points=('actual_cases', 'size'),
    mae_cases=('absolute_error_cases', 'mean'),
    bias_cases=('error_cases', 'mean'),
    mean_actual_cases=('actual_cases', 'mean'))
metrics['relative_mae'] = metrics.mae_cases / metrics.mean_actual_cases.clip(lower=1)
# Equal weight for each product and horizon when choosing one model family.
comparison = metrics.groupby('method', as_index=False).agg(
    mean_relative_mae=('relative_mae', 'mean'), tasks=('relative_mae', 'size'))
assert (comparison.tasks == len(products) * len(HORIZONS)).all()
selected_method = (comparison[comparison.method.isin(ML_METHODS)]
                   .sort_values(['mean_relative_mae', 'method']).iloc[0].method)
benchmark_score = comparison.loc[comparison.method == BENCHMARK, 'mean_relative_mae'].iloc[0]
selected_score = comparison.loc[comparison.method == selected_method, 'mean_relative_mae'].iloc[0]
beats_benchmark = bool(selected_score < benchmark_score)
display(metrics.round(3))
display(comparison.sort_values('mean_relative_mae').round(3))
print('Selected ML model family:', selected_method)
print('Better than the simple benchmark:', beats_benchmark)
if not beats_benchmark:
    print('Do not claim an ML improvement; explain this result to your supervisor.')


In [ ]:
# Keep the predictions, metrics, plots, configuration and fitted models in Drive.
predictions.to_csv(RUN_DIR / 'validation_predictions.csv', index=False)
shutil.copy2(METADATA_FILE, RUN_DIR / 'dataset_metadata.json')
metrics.to_csv(RUN_DIR / 'validation_metrics.csv', index=False)
comparison.to_csv(RUN_DIR / 'method_comparison.csv', index=False)
model_files = {}
for sku in products:
    values = data.loc[data.sku == sku].sort_values('date')['cases_ordered'].to_numpy(dtype=float)
    for horizon in HORIZONS:
        # After selection, refit the chosen family through validation only.
        origins = list(range(FIRST_ORIGIN, validation_last - horizon + 1, 7))
        x_all, y_all = examples(values, origins, horizon)
        fitted = make_model(selected_method).fit(x_all, y_all)
        filename = f'{sku}_{horizon}d.joblib'
        joblib.dump(fitted, RUN_DIR / 'models' / filename)
        model_files[f'{sku}_{horizon}'] = filename
        chart = predictions[(predictions.sku == sku) &
                            (predictions.horizon_days == horizon) &
                            (predictions.method.isin([BENCHMARK, selected_method]))]
        fig, ax = plt.subplots(figsize=(9, 3))
        actual = chart[chart.method == selected_method]
        ax.plot(actual.origin_date, actual.actual_cases, label='Actual', color='black')
        for method in (BENCHMARK, selected_method):
            subset = chart[chart.method == method]
            ax.plot(subset.origin_date, subset.predicted_cases, label=method)
        ax.set(title=f'{sku}: {horizon}-day validation', ylabel='Cases')
        ax.tick_params(axis='x', labelrotation=45)
        ax.legend()
        fig.tight_layout()
        fig.savefig(RUN_DIR / 'plots' / f'{sku}_{horizon}d.png', dpi=150)
        plt.close(fig)
manifest = {'run_id': RUN_ID, 'git_commit': GIT_COMMIT,
    'dataset_name': metadata['dataset_name'], 'dataset_sha256': data_hash,
    'dataset_is_synthetic': True, 'seed': SEED,
    'training_period': '2024-01-01 to 2025-09-30',
    'validation_period': '2025-10-01 to 2026-03-31',
    'excluded_later_period': '2026-04-01 to 2026-09-30',
    'prior_evaluation_warning': 'Earlier application runs may have used some later dates; do not claim an independent untouched test.',
    'selected_method': selected_method, 'beats_benchmark': beats_benchmark,
    'selection_rule': 'lowest mean relative MAE across four products and two horizons',
    'benchmark': 'previous-period case total; experiment only',
    'feature_names': ['last_day', 'lag_7', 'lag_14', 'lag_28', 'mean_7',
                      'mean_28', 'std_28', 'weekday_index'],
    'horizons_days': list(HORIZONS), 'model_files': model_files,
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
                 'joblib': joblib.__version__, 'matplotlib': matplotlib.__version__}}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved experiment evidence:', RUN_DIR)
print('Selected family:', selected_method, '| Later dates excluded from this run.')


## After the run
Open the Drive experiment folder and check the CSV metrics, prediction rows and plots. Save the completed notebook back to GitHub. Show the supervisor the split dates, the two ML models, the experiment-only benchmark, the selection rule, and the saved evidence. Do not claim real-company accuracy from this synthetic dataset.

The `.joblib` files contain the fitted version of the one selected algorithm for each product and horizon. AquaTrack does not load these files yet; the application must be connected to the chosen artifacts before it can be described as serving the selected trained model. Earlier application runs may have used some April-September 2026 dates, so do not present that period as an independent untouched test.